# Improved raw spectral clustering: one or many holes

This notebook generalises the useful part of `scripts/legacy/02_initial_cluster.ipynb`.

**What is clustered:** individual valid spectral readings, not whole holes.

**Why clusters are comparable:** every selected hole uses the same wavelength grid, and one shared SNV → PCA → K-Means model is fitted from a balanced training sample pooled across the selected holes.

Start with one hole if you want:

```python
HOLE_IDS = ["05GJD001"]
```

Then add holes without changing the workflow:

```python
HOLE_IDS = ["05GJD001", "MPWD10"]
```

SWIR/VNIR and TIR must be modelled separately. Change `REGION` and rerun the notebook to create the other model.


In [ ]:
# Imports and experiment settings

from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pytsg import parse_tsg
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Stable project location: this works regardless of the folder used to launch Jupyter.
REPO_ROOT = Path(
    r"C:\Users\DCCN9\OneDrive - UWA\Data Science"
    r"\4-Semester July-November 2026\Capstone\Data Crunch"
)

# Use one hole first, or list several holes to fit one shared model.
HOLE_IDS = ["05GJD001"]
# For a shared multi-hole model, use for example:
# HOLE_IDS = ["05GJD001", "MPWD10"]

# "nir" is the package's combined visible/NIR/SWIR array (531 bands here).
# Use "tir" for the separate thermal-infrared array (341 bands here).
REGION = "nir"

# Each hole contributes at most this many randomly selected spectra when fitting
# PCA and K-Means. All valid spectra receive a cluster after the model is fitted.
MAX_TRAINING_SPECTRA_PER_HOLE = 3_000

# K-Means settings. Candidate scores help us review this choice.
N_CLUSTERS = 4
K_CANDIDATES = range(2, 9)
RANDOM_STATE = 42

# Set True only when you want to write derived assignments under experiments/.
SAVE_OUTPUTS = False

assert REGION in {"nir", "tir"}
HOLE_IDS = list(dict.fromkeys(hole.upper() for hole in HOLE_IDS))
if not HOLE_IDS:
    raise ValueError("Specify at least one hole in HOLE_IDS.")

print("Holes:", HOLE_IDS)
print("Region:", REGION)


## 1. Load and quality-check each raw package

For every hole we:

1. locate its extracted TSG package;
2. select the requested spectral region;
3. keep readings where `Final Mask == 1` when that column exists;
4. remove spectra containing non-finite values or no spectral variation;
5. verify that every hole has the same wavelength grid.

A mismatch stops the notebook rather than silently combining incompatible spectra.


In [ ]:
def snv(values: np.ndarray) -> np.ndarray:
    """Standard Normal Variate: centre and scale each spectrum independently."""
    values = np.asarray(values, dtype=np.float64)
    row_mean = values.mean(axis=1, keepdims=True)
    row_std = values.std(axis=1, keepdims=True)
    if np.any(row_std <= 1e-12):
        raise ValueError("SNV received a flat or near-flat spectrum.")
    return (values - row_mean) / row_std


def load_hole(hole_id: str, region: str) -> dict:
    extracted = REPO_ROOT / "data" / "downloads" / "raw" / hole_id / "extracted"
    header = next(extracted.rglob("*_tsg.tsg"), None)
    if header is None:
        raise FileNotFoundError(
            f"No extracted TSG package found for {hole_id}. "
            f"Expected it below: {extracted}"
        )

    package = parse_tsg.read_package(header.parent, read_cras_file=False)
    section = getattr(package, region, None)
    if section is None or not hasattr(section, "spectra"):
        raise ValueError(f"{hole_id} has no {region.upper()} spectral section.")

    spectra_all = np.asarray(section.spectra, dtype=np.float64)
    wavelengths = np.asarray(section.wavelength, dtype=np.float64)
    headers = section.sampleheaders.reset_index(drop=True)
    scalars = section.scalars.reset_index(drop=True)

    if "D" not in headers.columns:
        raise KeyError(f"{hole_id} {region.upper()} sample headers have no depth column 'D'.")

    depths_all = pd.to_numeric(headers["D"], errors="coerce").to_numpy(dtype=float)
    if len(depths_all) != len(spectra_all):
        raise ValueError(f"{hole_id}: depth and spectral row counts do not match.")

    keep = np.isfinite(depths_all)
    mask_available = "Final Mask" in scalars.columns
    if mask_available:
        final_mask = pd.to_numeric(scalars["Final Mask"], errors="coerce").to_numpy()
        keep &= final_mask == 1

    keep &= np.isfinite(spectra_all).all(axis=1)
    keep &= spectra_all.std(axis=1) > 1e-12

    source_rows = np.flatnonzero(keep)
    spectra = spectra_all[keep]
    metadata = pd.DataFrame(
        {
            "hole_id": hole_id,
            "source_row": source_rows,
            "depth_m": depths_all[keep],
        }
    )

    return {
        "hole_id": hole_id,
        "spectra": spectra,
        "wavelengths": wavelengths,
        "metadata": metadata,
        "all_depths": depths_all,
        "total_readings": len(spectra_all),
        "mask_available": mask_available,
    }


datasets = {}
qc_rows = []
reference_wavelengths = None

for hole_id in HOLE_IDS:
    dataset = load_hole(hole_id, REGION)

    if reference_wavelengths is None:
        reference_wavelengths = dataset["wavelengths"]
    elif (
        len(dataset["wavelengths"]) != len(reference_wavelengths)
        or not np.allclose(dataset["wavelengths"], reference_wavelengths, atol=1e-6, rtol=0)
    ):
        raise ValueError(
            f"{hole_id} has a different {REGION.upper()} wavelength grid. "
            "Do not fit these holes in the same model without spectral resampling."
        )

    datasets[hole_id] = dataset
    qc_rows.append(
        {
            "hole_id": hole_id,
            "region": REGION.upper(),
            "total_readings": dataset["total_readings"],
            "valid_readings": len(dataset["spectra"]),
            "removed_readings": dataset["total_readings"] - len(dataset["spectra"]),
            "bands": dataset["spectra"].shape[1],
            "depth_min_m": dataset["metadata"]["depth_m"].min(),
            "depth_max_m": dataset["metadata"]["depth_m"].max(),
            "final_mask_available": dataset["mask_available"],
        }
    )

qc = pd.DataFrame(qc_rows)
display(qc)


## 2. Build a balanced shared training sample

A long hole should not dominate PCA or K-Means merely because it contains more readings. We therefore sample at most `MAX_TRAINING_SPECTRA_PER_HOLE` valid spectra from each hole for model fitting.

The fitted model is later applied to **all** valid readings. No valid reading is discarded from the final assignments.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
training_parts = []
training_metadata_parts = []

for hole_id, dataset in datasets.items():
    count = len(dataset["spectra"])
    training_count = min(count, MAX_TRAINING_SPECTRA_PER_HOLE)
    chosen = np.sort(rng.choice(count, size=training_count, replace=False))

    training_parts.append(snv(dataset["spectra"][chosen]))

    selected_metadata = dataset["metadata"].iloc[chosen].copy()
    selected_metadata["training_row"] = np.arange(training_count)
    training_metadata_parts.append(selected_metadata)

training_snv = np.vstack(training_parts)
training_metadata = pd.concat(training_metadata_parts, ignore_index=True)

print("Balanced training spectra:", f"{len(training_snv):,}")
display(training_metadata.groupby("hole_id").size().rename("training_spectra").to_frame())


## 3. Fit one shared PCA

PCA is dimensionality reduction, not clustering. It compresses the correlated wavelength bands while retaining at least 95% of the variance in the balanced, SNV-normalised training spectra.

Because one PCA is fitted across all selected holes, every hole is represented in the same component space.


In [ ]:
pca = PCA(n_components=0.95, svd_solver="full")
training_pca = pca.fit_transform(training_snv)

print("Original wavelength bands:", training_snv.shape[1])
print("PCA components retained:", pca.n_components_)
print(
    "Explained variance retained:",
    f"{pca.explained_variance_ratio_.sum() * 100:.2f}%",
)

pca_variance = pd.DataFrame(
    {
        "component": np.arange(1, pca.n_components_ + 1),
        "explained_variance_pct": pca.explained_variance_ratio_ * 100,
    }
)
display(pca_variance)


## 4. Review candidate values of K

K-Means requires the number of clusters in advance. Inertia always decreases as K increases, while a larger silhouette score indicates more separated clusters.

These diagnostics support a choice; they do not prove that clusters are geological units.


In [ ]:
candidate_rows = []

for k in K_CANDIDATES:
    if k >= len(training_pca):
        continue

    candidate = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=20,
    )
    candidate_labels = candidate.fit_predict(training_pca)

    silhouette_sample = min(5_000, len(training_pca))
    score = silhouette_score(
        training_pca,
        candidate_labels,
        sample_size=silhouette_sample,
        random_state=RANDOM_STATE,
    )
    candidate_rows.append(
        {
            "k": k,
            "inertia": candidate.inertia_,
            "silhouette": score,
        }
    )

candidate_scores = pd.DataFrame(candidate_rows)
display(candidate_scores)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(candidate_scores["k"], candidate_scores["inertia"], marker="o")
axes[0].set(title="Elbow diagnostic", xlabel="K", ylabel="Inertia")
axes[1].plot(candidate_scores["k"], candidate_scores["silhouette"], marker="o")
axes[1].set(title="Silhouette diagnostic", xlabel="K", ylabel="Silhouette score")
for axis in axes:
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 5. Fit the final shared K-Means model

All cluster IDs now refer to the same fitted centroids for every selected hole.


In [ ]:
if N_CLUSTERS < 2 or N_CLUSTERS >= len(training_pca):
    raise ValueError("N_CLUSTERS must be at least 2 and smaller than the training sample.")

kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=20,
)
training_labels = kmeans.fit_predict(training_pca)

training_counts = (
    pd.Series(training_labels, name="cluster_id")
    .value_counts()
    .sort_index()
    .rename("training_spectra")
    .to_frame()
)
display(training_counts)


## 6. Apply the shared model to every valid reading

The model was fitted on a balanced sample, but the following cell assigns a cluster to every valid spectrum in every selected hole.


In [ ]:
assignment_parts = []

for hole_id, dataset in datasets.items():
    hole_snv = snv(dataset["spectra"])
    hole_pca = pca.transform(hole_snv)
    labels = kmeans.predict(hole_pca)

    hole_assignments = dataset["metadata"].copy()
    hole_assignments["cluster_id"] = labels
    assignment_parts.append(hole_assignments)

assignments = (
    pd.concat(assignment_parts, ignore_index=True)
    .sort_values(["hole_id", "depth_m", "source_row"])
    .reset_index(drop=True)
)

print("Assigned valid spectra:", f"{len(assignments):,}")
display(assignments.head(20))


## 7. Compare cluster composition and depth patterns

The proportions show how frequently each global spectral cluster occurs in each hole. The depth plots show where those clusters occur downhole.

These are still spectral clusters, not mineral names and not gold labels.


In [ ]:
cluster_proportions = pd.crosstab(
    assignments["hole_id"],
    assignments["cluster_id"],
    normalize="index",
).mul(100)

cluster_proportions.columns = [
    f"cluster_{cluster_id}_pct" for cluster_id in cluster_proportions.columns
]
display(cluster_proportions.round(2))

fig, axes = plt.subplots(
    1,
    len(HOLE_IDS),
    figsize=(max(5, 4 * len(HOLE_IDS)), 9),
    squeeze=False,
)

for axis, hole_id in zip(axes[0], HOLE_IDS, strict=True):
    hole = assignments[assignments["hole_id"] == hole_id]
    axis.scatter(
        hole["cluster_id"],
        hole["depth_m"],
        c=hole["cluster_id"],
        cmap="tab10",
        s=3,
        alpha=0.5,
    )
    axis.set(
        title=hole_id,
        xlabel="Global cluster ID",
        ylabel="Depth (m)",
        xticks=range(N_CLUSTERS),
    )
    axis.invert_yaxis()
    axis.grid(alpha=0.15)

plt.suptitle(f"Shared {REGION.upper()} spectral clusters by depth", y=1.02)
plt.tight_layout()
plt.show()


## 8. Inspect the balanced mean spectral shape of each cluster

The following curves are cluster averages in SNV space. They help us see how the cluster shapes differ without allowing one long hole to dominate the average.


In [ ]:
plt.figure(figsize=(13, 6))

for cluster_id in range(N_CLUSTERS):
    cluster_mean = training_snv[training_labels == cluster_id].mean(axis=0)
    plt.plot(
        reference_wavelengths,
        cluster_mean,
        label=f"Cluster {cluster_id}",
        linewidth=1.5,
    )

plt.xlabel("Wavelength")
plt.ylabel("Mean SNV-normalised response")
plt.title(f"Balanced mean {REGION.upper()} spectral shape by global cluster")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 9. Optionally attach published processed TSG groups

Raw spectral clustering must not depend on TSG labels. When an all-logs processed Parquet exists and its depths align with the raw package, this cell attaches the most appropriate available group interpretation for comparison:

- NIR/SWIR: user `uTSAS`, then system `sTSAS`
- TIR: user `ujCLST`, then system `sjCLST`

Missing processed interpretations are reported and skipped rather than breaking the clustering.


In [ ]:
GROUP_CANDIDATES = {
    "nir": ["grp1_u_tsas", "grp1_s_tsas"],
    "tir": ["grp1_uj_clst", "grp1_sj_clst", "grp1_u_tsat", "grp1_s_tsat"],
}

assignments["tsg_group"] = pd.NA
assignments["tsg_group_source"] = pd.NA
comparison_messages = []

for hole_id, dataset in datasets.items():
    parquet_files = list(
        (
            REPO_ROOT
            / "data"
            / "downloads"
            / "processed"
            / hole_id
        ).glob("*/scalar_logs.parquet")
    )
    if not parquet_files:
        comparison_messages.append(f"{hole_id}: no processed Parquet; comparison skipped.")
        continue

    processed = pd.read_parquet(parquet_files[0]).reset_index(drop=True)
    group_column = next(
        (column for column in GROUP_CANDIDATES[REGION] if column in processed.columns),
        None,
    )
    if group_column is None:
        comparison_messages.append(
            f"{hole_id}: no compatible processed group log; comparison skipped."
        )
        continue

    processed_depth = pd.to_numeric(
        processed["start_depth"],
        errors="coerce",
    ).to_numpy(dtype=float)

    if (
        len(processed_depth) != len(dataset["all_depths"])
        or not np.allclose(
            processed_depth,
            dataset["all_depths"],
            atol=1e-4,
            rtol=0,
            equal_nan=True,
        )
    ):
        comparison_messages.append(
            f"{hole_id}: processed/raw depths do not align; comparison skipped."
        )
        continue

    target = assignments["hole_id"] == hole_id
    source_rows = assignments.loc[target, "source_row"].to_numpy(dtype=int)
    assignments.loc[target, "tsg_group"] = (
        processed.loc[source_rows, group_column].astype("string").to_numpy()
    )
    assignments.loc[target, "tsg_group_source"] = group_column
    comparison_messages.append(f"{hole_id}: attached {group_column}.")

print("\n".join(comparison_messages))

for hole_id in HOLE_IDS:
    comparison = assignments[
        (assignments["hole_id"] == hole_id)
        & assignments["tsg_group"].notna()
    ]
    if comparison.empty:
        continue

    table = pd.crosstab(
        comparison["cluster_id"],
        comparison["tsg_group"],
        normalize="index",
    ).mul(100)

    print(f"\n{hole_id}: TSG group percentages inside each spectral cluster")
    display(table.round(1))


## 10. Optional reproducible outputs

Saving is disabled by default. When enabled, the notebook writes only derived products; source raw packages and processed downloads remain unchanged.


In [ ]:
if SAVE_OUTPUTS:
    output_dir = REPO_ROOT / "experiments" / "03_improved_cluster"
    output_dir.mkdir(parents=True, exist_ok=True)

    assignments_path = output_dir / f"{REGION}_assignments.parquet"
    proportions_path = output_dir / f"{REGION}_hole_cluster_proportions.csv"
    settings_path = output_dir / f"{REGION}_settings.json"

    assignments.to_parquet(assignments_path, index=False)
    cluster_proportions.to_csv(proportions_path)

    settings = {
        "hole_ids": HOLE_IDS,
        "region": REGION,
        "maximum_training_spectra_per_hole": MAX_TRAINING_SPECTRA_PER_HOLE,
        "pca_components": int(pca.n_components_),
        "pca_explained_variance": float(pca.explained_variance_ratio_.sum()),
        "n_clusters": N_CLUSTERS,
        "random_state": RANDOM_STATE,
    }
    settings_path.write_text(json.dumps(settings, indent=2), encoding="utf-8")

    print("Saved:", assignments_path)
    print("Saved:", proportions_path)
    print("Saved:", settings_path)
else:
    print("SAVE_OUTPUTS is False; no derived files were written.")


## Interpretation limits

- Cluster IDs describe recurring spectral shapes; they are not mineral, lithology, or gold labels.
- PCA and K-Means are fitted once for the selected holes, so rerunning with a different selection can change the model.
- K-Means is a baseline that partitions continuous variation into a chosen number of groups.
- Nearby depth readings are spatially related; cluster counts are not independent observations.
- Compare clusters with TSG, geology, location, acquisition metadata, and chemistry before assigning geological meaning.
- Run NIR/SWIR and TIR as separate experiments.
